In [1]:
# %% [markdown]
# # Initial clustering run
# 
# This notebook will contain the initial clustering run with one fixed aggregation level and window size,
# choosing of optimal k, saving the model.

# %% [markdown]
# ## Data & Packages

# %%
# Main packages 
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

# Clustering packages
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score, adjusted_rand_score

# Parallel processing packages
from joblib import Parallel, delayed

# %%
# Load data
df = pl.scan_csv("/home/lanl/data/cyber1/auth.txt.gz", has_header=False, separator=",",
                 new_columns=['time', 'src_user', 'dest_user', 'src_comp', 'dest_comp',
                              'auth_type', 'logon_type', 'auth_orientation', 'outcome'])

# %%
# Keep only human users
df = df.filter(pl.col("src_user").str.starts_with("U"))

# %% [markdown]
# ## Functions & Global info

# %%
# Time conversions
SECONDS_IN_DAY = 60 * 60 * 24

# Time aggregation
agg_hour_level = 6

# %%
# FUNCTION - build features dataframe
def build_features(df, agg_hour_level):

    AGG_SECONDS = agg_hour_level * 60 * 60
    return (
        df.with_columns(
            bucket=pl.col('time') // AGG_SECONDS,
            theta=((pl.col('time') % SECONDS_IN_DAY) / SECONDS_IN_DAY) * 2 * np.pi,
            is_failure=(pl.col('outcome') == 'Fail').cast(pl.Int8),
        )
        .group_by(['src_user', 'bucket'])
        .agg(
            n_events=pl.len(),
            failure_ratio=pl.col('is_failure').mean(),
            n_distinct_dest=pl.col('dest_comp').n_unique(),
            n_distinct_src=pl.col('src_comp').n_unique(),
            c_bar=pl.col('theta').cos().mean(),
            s_bar=pl.col('theta').sin().mean(),
        )
        .with_columns(
            log_n_events=pl.col("n_events").log1p(),
            log_n_distinct_dest=pl.col("n_distinct_dest").log1p(),
            log_n_distinct_src=pl.col("n_distinct_src").log1p(),
        )
        .collect()
    )

# %%
# Relevant feature columns
feature_cols = [
    "log_n_events",
    "failure_ratio",
    "log_n_distinct_dest",
    "log_n_distinct_src",
    "c_bar",
    "s_bar",
]

# %%
# FUNCTION - process features for clustering 
def cluster_preprocess(features_df, feature_cols, agg_hour_level, week):

    buckets_per_week = (7 * 24) // agg_hour_level  # = 28 for 6-hour buckets
    lb = (week - 1) * buckets_per_week
    ub = lb + buckets_per_week - 1

    features_week = features_df.filter(pl.col('bucket').is_between(lb, ub))

    X = features_week.select(feature_cols).to_numpy()
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    return features_week, X_scaled

# %%
# FUNCTION - kmeans 
def fit_kmeans(k, Y, sample_size):
    km = KMeans(n_clusters=k, random_state=123, n_init=10)
    labels = km.fit_predict(Y)

    sil = silhouette_score(Y, labels, sample_size=sample_size, random_state=123)
    ch  = calinski_harabasz_score(Y, labels)
    db  = davies_bouldin_score(Y, labels)

    return k, sil, ch, db

# %%
# Create features dataframe
features_df = build_features(df, agg_hour_level)

# %% [markdown]
# ## Week 1 Analysis

# %%
# Process features for target week
features_w1, X_scaled = cluster_preprocess(features_df, feature_cols, agg_hour_level, week=1)

# %%
# Run kmeans in parallel
sample_size = 1000
results = Parallel(n_jobs=-1)(delayed(fit_kmeans)(k, X_scaled, sample_size) for k in range(2, 11))

# %%
# Best result
best_result = max(results, key=lambda row: row[1])
best_k, best_score = best_result[0], best_result[1]

print(f'optimal number of clusters is k = {best_k}')
print(f'silhouette score = {best_score}')

# %%
# Refit k-means on best result
km = KMeans(n_clusters=best_k, random_state=123, n_init=10)
labels = km.fit_predict(X_scaled)

# %%
# Attach labels back to the dataframe
features_w1 = (features_w1
    .with_columns(pl.Series("cluster", labels))
    .select(['src_user', 'bucket', 'cluster'])
)

# %%
# Cluster breakdown
features_w1['cluster'].value_counts()

# %%
# Save results
features_w1.write_parquet("/home/ma/t/tr725/MSc-Project/03_working_notebooks/results/week1_6h.parquet")

# %% [markdown]
# ## Week 2 Analysis

# %%
# Process features for target week
features_w2, X_scaled = cluster_preprocess(features_df, feature_cols, agg_hour_level, week=2)

# %%
# Run kmeans in parallel
sample_size = 1000
results = Parallel(n_jobs=-1)(delayed(fit_kmeans)(k, X_scaled, sample_size) for k in range(2, 11))

# %%
# Best result
best_result = max(results, key=lambda row: row[1])
best_k, best_score = best_result[0], best_result[1]

print(f'optimal number of clusters is k = {best_k}')
print(f'silhouette score = {best_score}')

# %%
# Refit k-means on best result
km = KMeans(n_clusters=best_k, random_state=123, n_init=10)
labels = km.fit_predict(X_scaled)

# %%
# Attach labels back to the dataframe
features_w2 = (features_w2
    .with_columns(pl.Series("cluster", labels))
    .select(['src_user', 'bucket', 'cluster'])
)

# %%
# Cluster breakdown
features_w2['cluster'].value_counts()

# %%
# Save results
features_w2.write_parquet("/home/ma/t/tr725/MSc-Project/03_working_notebooks/results/week2_6h.parquet")

# %% [markdown]
# ## Adjusted Rand Index

# %%
buckets_per_week = (7 * 24) // agg_hour_level  # = 28

features_w1 = features_w1.with_columns(
    rel_bucket=pl.col('bucket') % buckets_per_week
)

features_w2 = features_w2.with_columns(
    rel_bucket=pl.col('bucket') % buckets_per_week
)

overlap = features_w1.join(
    features_w2,
    on=['src_user', 'rel_bucket'],
    how='inner',
    suffix='_w2'
)

# %%
labels_w1 = overlap["cluster"].to_numpy()
labels_w2 = overlap["cluster_w2"].to_numpy()

ari = adjusted_rand_score(labels_w1, labels_w2)
print(ari)

# %%
# Relative (user, 6h-bucket-within-week) overlap between week 1 and week 2
features_w1, _ = cluster_preprocess(features_df, feature_cols, agg_hour_level, week=1)
features_w2, _ = cluster_preprocess(features_df, feature_cols, agg_hour_level, week=2)

# Add relative bucket (6h-bucket within week) to each
rel_w1 = features_w1.with_columns(
    (pl.col("bucket") % buckets_per_week).alias("rel_bucket")
).select(["src_user", "rel_bucket"])

rel_w2 = features_w2.with_columns(
    (pl.col("bucket") % buckets_per_week).alias("rel_bucket")
).select(["src_user", "rel_bucket"])

# Inner join to find overlapping (user, rel_bucket) pairs
overlap = rel_w1.join(rel_w2, on=["src_user", "rel_bucket"], how="inner")

print(f"Week 1 (user, rel_bucket) pairs:  {len(rel_w1)}")
print(f"Week 2 (user, rel_bucket) pairs:  {len(rel_w2)}")
print(f"Overlapping pairs:                {len(overlap)}")
print(f"Overlap as % of week 1:           {len(overlap)/len(rel_w1):.2%}")
print(f"Overlap as % of week 2:           {len(overlap)/len(rel_w2):.2%}")

optimal number of clusters is k = 2
silhouette score = 0.4484713291708299
optimal number of clusters is k = 2
silhouette score = 0.4506224684526274
0.7387094444306598
Week 1 (user, rel_bucket) pairs:  179408
Week 2 (user, rel_bucket) pairs:  199818
Overlapping pairs:                141712
Overlap as % of week 1:           78.99%
Overlap as % of week 2:           70.92%
